In [2]:
from langchain import messages
from langchain.agents import AgentState, create_agent
from langchain_core.messages import ToolMessage
from langgraph.checkpoint.memory import InMemorySaver


class CustomState(AgentState):
    favourite_colour:str

In [1]:
from json import tool

# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

True

In [5]:

from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour,   # Write to custom state field
        "messages": [                            # Must manually include ToolMessage
            ToolMessage(
                "Successfully updated favourite colour",
                tool_call_id=runtime.tool_call_id  # Links to the AI's tool call request
            )
        ]
    })

In [6]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
agent=create_agent("groq:llama-3.1-8b-instant",
                   tools=[update_favourite_colour],
                   checkpointer=InMemorySaver(),
                   state_schema=CustomState)

In [13]:
from langchain.messages import HumanMessage
from pprint import pprint
response=agent.invoke(
    {"messages":[HumanMessage(content="My favourite colour is green")]},
    {"configurable":{"thread_id":"1"}}
)
pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='dee908ca-97d6-4bc6-a78a-b65a093ae86e'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5nkmc7w6t', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 243, 'total_tokens': 262, 'completion_time': 0.021290061, 'completion_tokens_details': None, 'prompt_time': 0.015910963, 'prompt_tokens_details': None, 'queue_time': 0.055373676, 'total_time': 0.037201024}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f93ef-7d94-7d32-bd3b-06f3152d354c-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'i

In [8]:
@tool
def update_only_msg():
    """updating custom fileds"""
    return "updated the tool msg state"

In [9]:
response=agent.invoke({
    "messages":[HumanMessage(content="hello how are you?")],
    "favourite_colour":"green"
},
    {"configurable":{"thread_id":"10"}}
)
pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='hello how are you?', additional_kwargs={}, response_metadata={}, id='be60e3ff-fdf0-4c78-af29-759f0eb9361f'),
              AIMessage(content="I'm functioning properly, thank you for asking. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 243, 'total_tokens': 261, 'completion_time': 0.029267297, 'completion_tokens_details': None, 'prompt_time': 0.023383556, 'prompt_tokens_details': None, 'queue_time': 0.061376793, 'total_time': 0.052650853}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f93ef-9eed-7d22-8ffa-63d7b4437688-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 243, 'output_tokens': 18, 'total_tokens': 261})]}


In [10]:
@tool
def read_favourite_colur(runtime: ToolRuntime) -> str:
    """Read the favourite colour from the user."""
    try:
        return runtime.state['favourite_colour']
    except KeyError:
        return "No favourite colour found"

In [12]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}  # Same thread — state is loaded
)

pprint(response)


{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='dee908ca-97d6-4bc6-a78a-b65a093ae86e'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5nkmc7w6t', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 243, 'total_tokens': 262, 'completion_time': 0.021290061, 'completion_tokens_details': None, 'prompt_time': 0.015910963, 'prompt_tokens_details': None, 'queue_time': 0.055373676, 'total_time': 0.037201024}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f93ef-7d94-7d32-bd3b-06f3152d354c-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'i